<a href="https://colab.research.google.com/github/catseyehuang/Taipei-YouBike-Dashboard/blob/main/YouBike_JSON.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# YouBike : ETL 流程

## 1. 擷取 (Extract):


### JSON 檔案讀取

In [7]:
import os
import json
import pandas as pd
from google.colab import drive
import re

In [8]:
# ==========================================
# 🚀 Data Lake JSON 批次讀取
# ==========================================

# 1. 掛載您的 Google Drive (執行時會跳出視窗要求授權)
print("🔗 正在要求授權掛載 Google Drive...")
drive.mount('/content/drive')

# 2. 指定 Data Lake 資料夾路徑
folder_path = '/content/drive/MyDrive/C02_AI/Vibe Coding/YouBike/YouBike_Raw_JSON'

all_records = []
print(f"📂 正在掃描資料夾：{folder_path}")

# 3. 掃描並解析資料夾內的所有 JSON 檔案
if not os.path.exists(folder_path):
    print(f"❌ 找不到路徑 {folder_path}，請檢查資料夾名稱是否正確！")
else:
    file_list = [f for f in os.listdir(folder_path) if f.endswith(".json")]
    print(f"🔍 共找到 {len(file_list)} 個 JSON 快照檔案，執行下一步開始進行清洗與對齊...")

🔗 正在要求授權掛載 Google Drive...
Mounted at /content/drive
📂 正在掃描資料夾：/content/drive/MyDrive/C02_AI/Vibe Coding/YouBike/YouBike_Raw_JSON
🔍 共找到 876 個 JSON 快照檔案，執行下一步開始進行清洗與對齊...


###  JSON 檔案數量統計 (7x24 網格)

In [11]:
# ==========================================
# 🚀 建立獨立的 df_file_records 支援 7x24 網格
# ==========================================

if not os.path.exists(folder_path):
    print(f"❌ 找不到路徑 {folder_path}，請檢查資料夾名稱是否正確！")
else:
    # 1. 掃描資料夾內所有的 JSON 快照檔案
    json_files = [f for f in os.listdir(folder_path) if f.endswith(".json")]

    file_records = []
    for filename in json_files:
        match = re.match(r'^(TPE|NTPC)_(\d{4})(\d{2})(\d{2})_(\d{2})\d{4}\.json$', filename)
        if match:
            city_code, year, month, day, hour = match.groups()

            city = 'Taipei' if city_code == 'TPE' else 'New Taipei'
            scrape_date = f"{year}-{month}-{day}"
            scrape_hour = int(hour)

            file_records.append({
                'source_file': filename,
                'city': city,
                'scrape_date': scrape_date,
                'scrape_hour': scrape_hour
            })

    df_file_records = pd.DataFrame(file_records)
    print(f"✅ 成功建立獨立的 df_file_records！共偵測到 {len(df_file_records)} 個檔案。")

    if len(df_file_records) > 0:
        # 2. 將日期轉為 datetime 格式，以萃取出「星期」
        df_file_records['scrape_date'] = pd.to_datetime(df_file_records['scrape_date'])

        # 建立星期對應表 (0=週一, 6=週日)
        weekday_map = {0: '週一', 1: '週二', 2: '週三', 3: '週四', 4: '週五', 5: '週六', 6: '週日'}
        df_file_records['weekday'] = df_file_records['scrape_date'].dt.dayofweek.map(weekday_map)

        # 3. 🎯 核心邏輯：計算每個 (城市, 星期, 小時) 組合下，有幾個「不重複的日期 (scrape_date)」
        # 這樣同一天同小時有多個檔案只算 1，但跨週的不同日期就會累加！
        weekly_hourly_counts = df_file_records.groupby(['city', 'weekday', 'scrape_hour'])['scrape_date'].nunique()

        # 將城市 (city) 展開
        df_unstacked = weekly_hourly_counts.unstack(level='city', fill_value=0)

        # 4. 建立完整的 7x24 網格索引，補齊沒有資料的小時與星期
        weekday_order = ['週一', '週二', '週三', '週四', '週五', '週六', '週日']
        all_hours = list(range(24))
        full_idx = pd.MultiIndex.from_product([weekday_order, all_hours], names=['weekday', 'scrape_hour'])

        # 重新索引以填補 0
        df_full_grid = df_unstacked.reindex(full_idx, fill_value=0)

        # ==========================================
        # 📊 呈現雙北分開的 7x24 檔案收集天數統計表 (轉置：星期為列，小時為欄)
        # ==========================================
        print("\n" + "="*45)
        print("📈 新北市 JSON 檔案收集天數統計 (7x24 星期網格)")
        print("="*45)
        # 將 scrape_hour 展開成上方欄位，並使用 reindex 強制左側列照星期一到日排序
        ntpc_pivot = df_full_grid['New Taipei'].unstack(level='scrape_hour', fill_value=0)
        ntpc_pivot = ntpc_pivot.reindex(weekday_order)
        display(ntpc_pivot)

        print("\n" + "="*45)
        print("📈 台北市 JSON 檔案收集天數統計 (7x24 星期網格)")
        print("="*45)
        tpe_pivot = df_full_grid['Taipei'].unstack(level='scrape_hour', fill_value=0)
        tpe_pivot = tpe_pivot.reindex(weekday_order)
        display(tpe_pivot)

    else:
        print("⚠️ 資料夾內無符合命名規則的 JSON 檔案。")


✅ 成功建立獨立的 df_file_records！共偵測到 878 個檔案。

📈 新北市 JSON 檔案收集天數統計 (7x24 星期網格)


scrape_hour,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,19,20,21,22,23
weekday,,,,,,,,,,,,,,,,,,,,,
週一,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
週二,0,0,0,0,0,0,0,0,0,0,...,1,1,1,1,1,1,1,1,1,1
週三,1,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,1
週四,1,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,0,0,0,0
週五,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
週六,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
週日,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0



📈 台北市 JSON 檔案收集天數統計 (7x24 星期網格)


scrape_hour,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,19,20,21,22,23
weekday,,,,,,,,,,,,,,,,,,,,,
週一,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
週二,0,0,0,0,0,0,0,0,0,0,...,1,1,1,1,1,1,1,1,1,1
週三,1,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,1
週四,1,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,0,0,0,0
週五,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
週六,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
週日,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## 2. 轉換 (Transform)

### 建立初始 DataFrame (去重複)

- 將新北市與台北市 雙北資料標準化後，合併進入同一個 DataFrame
- 根據 sno(站點編號) + update_time(站點資料更新時間)去除重複資料

In [2]:
# ==========================================
# 🚀 建立 DataFrame
# ==========================================

# 1. 掃描並解析資料夾內的所有 JSON 檔案
if not os.path.exists(folder_path):
    print(f"❌ 找不到路徑 {folder_path}，請檢查資料夾名稱是否正確！")
else:
    file_list = [f for f in os.listdir(folder_path) if f.endswith(".json")]
    print(f"🔍 共找到 {len(file_list)} 個 JSON 快照檔案，開始進行清洗與對齊...")

    for filename in file_list:
        file_path = os.path.join(folder_path, filename)

        with open(file_path, 'r', encoding='utf-8') as f:
            try:
                data = json.load(f)

                # 🛡️ 處理新北市格式 (NTPC)
                if filename.startswith("NTPC_"):
                    for item in data:
                        all_records.append({
                            'city': 'New Taipei',
                            'sno': item.get('sno', ''),
                            'sna': item.get('sna', ''),
                            'sarea': item.get('sarea', ''),
                            'tot_quantity': int(item.get('tot_quantity', '' )),
                            'sbi_quantity': int(item.get('sbi_quantity', '')),
                            'bemp': int(item.get('bemp', '')),
                            'lat': float(item.get('lat', '')),
                            'lng': float(item.get('lng', '')),
                            'update_time': item.get('mday',''),
                            'is_active': int(item.get('act', '')),
                            'source_file': filename # 追蹤來源檔案
                        })

                # 🛡️ 處理台北市格式 (TPE)
                elif filename.startswith("TPE_"):
                    for item in data:
                        all_records.append({
                            'city': 'Taipei',
                            'sno': item.get('sno', ''),
                            'sna': item.get('sna', ''),
                            'sarea': item.get('sarea', ''),
                            'tot_quantity': int(item.get('Quantity', '')),
                            'sbi_quantity': int(item.get('available_rent_bikes', '')),
                            'bemp': int(item.get('available_return_bikes', '')),
                            'lat': float(item.get('latitude', '')),
                            'lng': float(item.get('longitude', '')),
                            'update_time': item.get('infoTime', ''),
                            'is_active': int(item.get('act', '')),
                            'source_file': filename
                        })
            except Exception as e:
                print(f"⚠️ 解析檔案 {filename} 發生錯誤: {e}")

    # 2. 轉換為 Pandas DataFrame
    if len(all_records) > 0:
        df_history = pd.DataFrame(all_records)

        # 時間格式標準化 (將字串轉為 datetime 物件，方便後續畫圖)
        df_history['update_time'] = pd.to_datetime(df_history['update_time'], format='mixed', errors='coerce')

        # === 以下為新增與修正的欄位處理，確保名稱與格式與用戶需求一致 ===

        # 1. scrape_timestamp (datetime object: 'YYYY-MM-DD HH:MM:SS')
        #    先從 source_file 欄位中提取日期時間字串 (e.g., '20260617_172106')
        # 修正 regex，將 \d 和 \. 改為單個 \d 和 \. 以正確匹配
        temp_scrape_dt_str = df_history['source_file'].str.extract(r'_(\d{8}_\d{6})\.json', expand=False)
        #    將提取的字串轉換為 datetime 物件
        df_history['scrape_timestamp'] = pd.to_datetime(temp_scrape_dt_str, format='%Y%m%d_%H%M%S', errors='coerce')

        # 2. scrape_date (date object: 'YYYY-MM-DD')
        #    從 scrape_timestamp 提取日期部分
        df_history['scrape_date'] = df_history['scrape_timestamp'].dt.date

        # 3. date (date object: 'YYYY-MM-DD', 來自 update_time)
        #    從 update_time 欄位中提取日期部分
        df_history['date'] = df_history['update_time'].dt.date

        # 4. hour (int: '0-23', 來自 update_time)
        #    從 update_time 欄位中提取小時部分
        df_history['hour'] = df_history['update_time'].dt.hour

        # 5. weekday (int: '0-6', 來自 update_time, 0=Monday)
        #    從 update_time 欄位中提取星期幾部分
        df_history['weekday'] = df_history['update_time'].dt.dayofweek

        # 清理可能存在的舊有或不一致的欄位
        # 移除 'file_date' 和 'file_timestamp' 等不再需要的欄位，避免混淆
        df_history = df_history.drop(columns=['file_date', 'file_timestamp'], errors='ignore')

        # 依照時間與站點排序，確保時序正確 (保留原有的註釋行，不啟用)
        #df_history = df_history.sort_values(by=['update_time', 'sno']).reset_index(drop=True)

        print(f"\n✅ 成功將 JSON 轉換為 DataFrame！共載入 {len(df_history)} 筆資料。")

        # 顯示資料全貌與前五筆
        print("="*45)
        display(df_history.info())
        print("="*45)
        display(df_history.head())
    else:
        print("\n⚠️ 目錄中沒有符合格式的資料，請確認 GAS 排程是否有成功下載檔案。")

🔍 共找到 2 個 JSON 快照檔案，開始進行清洗與對齊...

✅ 成功將 JSON 轉換為 DataFrame！共載入 3319 筆資料。
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3319 entries, 0 to 3318
Data columns (total 17 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   city              3319 non-null   object        
 1   sno               3319 non-null   object        
 2   sna               3319 non-null   object        
 3   sarea             3319 non-null   object        
 4   tot_quantity      3319 non-null   int64         
 5   sbi_quantity      3319 non-null   int64         
 6   bemp              3319 non-null   int64         
 7   lat               3319 non-null   float64       
 8   lng               3319 non-null   float64       
 9   update_time       3319 non-null   datetime64[ns]
 10  is_active         3319 non-null   int64         
 11  source_file       3319 non-null   object        
 12  scrape_timestamp  3319 non-null   datetime64[ns]
 13  scrap

None

,city,sno,sna,sarea,tot_quantity,sbi_quantity,bemp,lat,lng,update_time,is_active,source_file,scrape_timestamp,scrape_date,date,hour,weekday
0,Taipei,500101001,YouBike2.0_捷運科技大樓站,大安區,28,6,21,25.02605,121.54360,2026-06-17 00:32:03,1,TPE_20260617_004105.json,2026-06-17 00:41:05,2026-06-17,2026-06-17,0,2
1,Taipei,500101002,YouBike2.0_復興南路二段273號前,大安區,21,11,10,25.02565,121.54357,2026-06-17 00:00:03,1,TPE_20260617_004105.json,2026-06-17 00:41:05,2026-06-17,2026-06-17,0,2
2,Taipei,500101003,YouBike2.0_國北教大實小東側門,大安區,28,22,5,25.02429,121.54124,2026-06-17 00:08:02,1,TPE_20260617_004105.json,2026-06-17 00:41:05,2026-06-17,2026-06-17,0,2
3,Taipei,500101004,YouBike2.0_和平公園東側,大安區,11,8,1,25.02351,121.54282,2026-06-17 00:36:03,1,TPE_20260617_004105.json,2026-06-17 00:41:05,2026-06-17,2026-06-17,0,2
4,Taipei,500101005,YouBike2.0_辛亥復興路口西北側,大安區,16,8,8,25.02153,121.54299,2026-06-17 00:40:03,1,TPE_20260617_004105.json,2026-06-17 00:41:05,2026-06-17,2026-06-17,0,2


### 資料淨化：建立黑名單與幽靈站點

# 3. 全域總覽儀表板 (Macro Overview Dashboard)

### 探索性資料分析 (EDA - Exploratory Data Analysis

### 雙北 YouBike 2.0 站點容量資源分佈地圖 (Station Capacity Map)

從三個空間維度來看出故事：

- 交通樞紐的「巨型大動脈」：
那些面積大到誇張的藍色圓圈，通常會極度精準地黏在捷運轉乘站、台鐵/高鐵站、以及大學城（如台大公館生活圈）的周邊。這代表微笑單車在都市計畫中，是扮演標準的「大眾運輸最後一哩路（Last-Mile Connector）」。

- 社區內部的「微血管分佈」：
相較之下，住宅區、巷弄、小公園周邊則佈滿了非常密集、但藍圈極小的「微型站（10~15 柱）」。這能展現 YouBike 2.0 無須牽電線的靈活特性，成功把服務觸角塞進城市的死角。

- 調度與營運瓶頸預警（Bottleneck Alert）：
藍色圓圈越大，代表它在通勤尖峰時間能吐出、或吞下的車流量越驚人！ 這張圖直接點名了哪些站點是未來的「調度一線戰場」。這些大圈圈只要一壞掉或被塞滿，全網的流動性就會瞬間卡死。